# 在 Colab 用 webcam 跑 YOLO（瀏覽器鏡頭版）

`yolo_only.ipynb` 的第 4 格是**純文字說明**（不是程式碼），所以點它不會有反應。
原因很單純：**Colab 的 VM 在 Google 雲端，那台機器上沒有實體相機。**

```python
cap = cv2.VideoCapture(0)
cap.isOpened()      # → False，永遠
```

裝任何套件都改變不了「雲端機器沒有鏡頭」。要用 webcam，只有兩條路：

| 方案 | 在哪跑 | 說明 |
|---|---|---|
| **A. 回自己的電腦**（推薦） | 本機終端機 | 真即時、跑滿 30 FPS，考試就是這樣 |
| **B. 這本 notebook** | Colab 瀏覽器 | 用 JavaScript 抓**你瀏覽器**的鏡頭，逐張送回 Python |

## 方案 B 的前提

- ✅ 必須用 **Chrome / Edge**（Safari、手機通常不行）
- ✅ 網址必須是 `colab.research.google.com`
- ✅ 彈出授權時要按「**允許**」使用鏡頭
- ⚠️ 這是**逐張抓拍**，不是即時串流；速度受瀏覽器往返影響，比本機慢很多
- ⚠️ **不要拿這裡的 FPS 當效能指標**（要量效能請回本機）


## 1. 安裝


In [ ]:
%pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)


## 2. 啟動瀏覽器鏡頭

跑這格後，下方會出現一個影像框並詢問鏡頭權限。**按「允許」**。

若沒彈出提示，請點瀏覽器網址列的 🔒／相機圖示 → 把「相機」設為允許 → 重跑這格。


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

# NOTE: eval_js evaluates each payload in a fresh scope, so a helper defined in
# one call is gone by the next one. All state therefore lives on window.__ivdc
# and every payload is fully self-contained.
_init_js = """
(async () => {
  const S = window.__ivdc = window.__ivdc || {video: null, stream: null, canvas: null, div: null};

  if (!S.canvas) { S.canvas = document.createElement('canvas'); }

  if (S.stream) { return 'already running'; }

  S.div = document.createElement('div');
  S.video = document.createElement('video');
  S.video.style.display = 'block';
  S.video.width = 640;
  S.video.setAttribute('playsinline', '');
  S.div.appendChild(S.video);
  document.body.appendChild(S.div);

  S.stream = await navigator.mediaDevices.getUserMedia({video: true});
  S.video.srcObject = S.stream;
  await S.video.play();

  return 'ready';
})()
"""

try:
    state = eval_js(_init_js)
    print("鏡頭已啟動:", state)
except Exception as exc:
    print("鏡頭啟動失敗 ->", type(exc).__name__, exc)
    print()
    print("請確認：")
    print("  1. 用 Chrome 或 Edge（Safari/手機通常不行）")
    print("  2. 網址是 colab.research.google.com")
    print("  3. 彈出授權時按「允許」；若已拒絕，點網址列鎖頭圖示改成允許後重跑本格")


## 3. 抓一張，確認畫面真的有收到


In [ ]:
from IPython.display import Image, display


def grab_frame():
    """Grab one frame from the browser camera, return a BGR ndarray."""
    js = """
    (() => {
      const S = window.__ivdc;
      if (!S || !S.video) { throw new Error('camera not started - run the init cell first'); }
      S.canvas.width = S.video.videoWidth;
      S.canvas.height = S.video.videoHeight;
      S.canvas.getContext('2d').drawImage(S.video, 0, 0);
      return S.canvas.toDataURL('image/jpeg', 0.8);
    })()
    """
    data = eval_js(js)
    binary = b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)


frame = grab_frame()
print("frame shape:", None if frame is None else frame.shape)
print("brightness:", None if frame is None else round(float(frame.mean()), 1))
cv2.imwrite("colab_webcam_probe.jpg", frame)
display(Image("colab_webcam_probe.jpg", width=480))


## 4. 用 YOLO 跑鏡頭畫面（抓 10 幀）

`stream=False` + 逐張 predict，是因為這裡是抓拍模式，不是連續串流。


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # 第一次會下載 ~6 MB

N = 10
last_result = None
for i in range(N):
    frame = grab_frame()
    if frame is None:
        print(f"frame {i}: 抓不到畫面，跳過")
        continue
    r = model.predict(frame, imgsz=640, conf=0.25, verbose=False)[0]
    n = len(r.boxes)
    top = float(r.boxes.conf.max()) if n else 0.0
    print(f"frame {i:02d}: {n} 個物件, 最高信心 {top:.2f}, 推論 {r.speed['inference']:.0f} ms")
    last_result = r

if last_result is not None:
    last_result.save(filename="colab_webcam_last.jpg")
    print("\n最後一幀 → colab_webcam_last.jpg")
    display(Image("colab_webcam_last.jpg", width=640))


## 5. 沒偵測到東西？

**幾乎都不是程式的問題：**

- COCO 模型只認 **80 類真實物體**（person / cup / bottle / cell phone / laptop / keyboard…）
- 對著**牆壁、天花板、純色** → 0 個很正常
- 光線太暗 → 提高 `--conf` 之前先開燈；或把 `conf=0.25` 降到 `0.15`

**拿這些給它看**：杯子、瓶子、手機、鍵盤、馬克杯、自己的臉（person）。

## 6. 真的即時偵測 → 回自己的電腦

Colab 這條路是抓拍，不是即時。要真正的即時畫面，回本機終端機：

```powershell
cd C:\Users\princ\ivdc_handout
.\.venv-yolo\Scripts\python.exe src\05_yolo_detection.py --source 0 --imgsz 640
```

按 `q` 或關視窗結束。腳本會**自動掃描 0~6 並量亮度**，跳過黑屏的虛擬裝置 ——
所以**不要手寫 `--source 2`**，索引每次開機都可能變。

## 7. 在 Colab 不用鏡頭的替代方案

| 需求 | 做法 |
|---|---|
| 只是想證明「即時偵測」流程能跑 | `yolo_only.ipynb` 第 3 格（用 `bus.jpg` 合成 `clip.mp4`） |
| 想用自己的影片 | 左側檔案面板上傳 `.mp4` → `model.predict(source="my.mp4", stream=True)` |
| 想量真實 FPS | 一定要回本機，Colab 的抓拍模式量不準 |
